# 11 — CIFAR-10N / CIFAR-100N: fine-tune and extract out-of-sample predictions

**Purpose (reviewer response, independent dataset).** The paper's "when does group-level
uncertainty help" section predicts that the group signal becomes informative as the *effective
label space shrinks* and the *classifier sharpens*. On AlleNoise that was shown only by
restricting the stored predictions internally (nb08). This notebook supplies the **independent**
evidence a reviewer asked for: two genuinely small-label-space benchmarks with **real human
annotation noise** — CIFAR-10N (K=10) and CIFAR-100N (K=100) — trained from scratch on the noisy
human labels, with clean labels held aside only for evaluation.

It mirrors `01_finetune_and_predict.ipynb`: train on the **noisy** labels, obtain
**out-of-sample** predicted distributions \(p_i\) by k-fold cross-fitting (each item predicted by
a model that never trained on it), and save artifacts in the **same schema** so the analysis
notebook (nb12) can read them exactly like nb06/07/08 read AlleNoise.

**You run this.** Needs a GPU. It downloads CIFAR-10/100 (torchvision) and the CIFAR-N human-noise
label files. Set `DATASET` in the config cell to `"cifar10n"` then `"cifar100n"` and run twice.


### Setup — install dependencies (run once)
If `torch` / `torchvision` are missing (e.g. a fresh runtime), run this cell first. On a GPU
runtime it installs the CUDA build; on CPU it installs the CPU build. If torch is installed for
the first time, restart the kernel once and re-run from here. Skip if your environment already has
them (Colab GPU does).

In [1]:
# --- dependency check / install (safe to re-run) ---
import importlib, subprocess, sys
def _have(m):
    try: importlib.import_module(m); return True
    except Exception: return False

need = [m for m in ("torch","torchvision","sklearn","pandas","scipy","seaborn","pyarrow") if not _have(m)]
if need:
    print("installing:", need)
    pkgs=[{"sklearn":"scikit-learn"}.get(m,m) for m in need]
    subprocess.run([sys.executable,"-m","pip","install","-q",*pkgs], check=True)
    print("done. If torch/torchvision were just installed, RESTART the kernel once, then re-run from here.")
else:
    print("all dependencies present")

import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| cuda available:", torch.cuda.is_available())

all dependencies present
torch 2.6.0+cu124 | torchvision 0.21.0+cu124 | cuda available: True


In [2]:
# Notebook: 11_cifar_finetune_predict
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions (matches nb01).
import os, time, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8
plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)
def savefig(fig,name):
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG,f"{name}.{ext}"),dpi=600,bbox_inches="tight")
    plt.close(fig)

In [3]:
# --- config ---
import torch
# >>> SET THIS, run the whole notebook, then set to the other and run again <<<
DATASET   = "cifar100n"        # "cifar10n" (K=10)  or  "cifar100n" (K=100)

# CIFAR-N noise variants: cifar10n uses the "aggre" label by default (worst/rand1 also available);
# cifar100n has a single "noisy_label". Changing CIFAR10N_NOISE lets you probe a higher noise rate.
CIFAR10N_NOISE = "aggre_label"   # one of: aggre_label, random_label1/2/3, worse_label
DATA_ROOT  = os.path.join("..","data","cifar")     # torchvision + CIFAR-N files live here
OUT_DIR    = os.path.join("..","data","cifar", DATASET)   # artifacts for THIS dataset
os.makedirs(DATA_ROOT, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

N_FOLDS    = 3          # OOS cross-fit; 3 folds suffice (matches nb01) and is faster
EPOCHS     = 30          # cosine; CIFAR-N converges by ~30, and stopping early avoids memorizing noise
LR         = 0.1         # SGD + cosine (standard CIFAR recipe)
BATCH      = 256         # larger batch -> fewer steps on GPU
USE_AMP    = True        # mixed precision: ~1.5x faster on GPU, negligible accuracy impact
WEIGHT_DECAY = 5e-4
MOMENTUM   = 0.9
LABEL_SMOOTH = 0.0       # keep 0: we must not artificially sharpen/soften beliefs here
SEED       = 42
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| dataset:", DATASET, "| noise:", CIFAR10N_NOISE if DATASET=="cifar10n" else "noisy_label")
if DEVICE != "cuda":
    print("[warning] CUDA not detected -> running on CPU. CIFAR + ResNet-34 is very slow on CPU;")
    print("          enable a GPU runtime for paper-quality results. Proceeding anyway.")

import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device: cuda | dataset: cifar100n | noise: noisy_label


In [4]:
# --- get CIFAR-N human-noise labels ---
# CIFAR-N (Wei et al., ICLR 2022) provides REAL human annotation noise for CIFAR-10/100.
# Files: CIFAR-10_human.pt (keys: clean_label, aggre_label, worse_label, random_label1..3),
#        CIFAR-100_human.pt (keys: clean_label, noisy_label, noisy_coarse_label).
# They are distributed in the CIFAR-N GitHub release. Download may be blocked in some
# environments; if so, drop the two .pt files into DATA_ROOT manually and re-run this cell.
import urllib.request, os
CIFARN_URLS = {
    "CIFAR-10_human.pt":  "https://github.com/UCSC-REAL/cifar-10-100n/raw/main/data/CIFAR-10_human.pt",
    "CIFAR-100_human.pt": "https://github.com/UCSC-REAL/cifar-10-100n/raw/main/data/CIFAR-100_human.pt",
}
need = "CIFAR-10_human.pt" if DATASET=="cifar10n" else "CIFAR-100_human.pt"
dst = os.path.join(DATA_ROOT, need)
if not os.path.exists(dst):
    try:
        print("downloading", need, "...")
        urllib.request.urlretrieve(CIFARN_URLS[need], dst)
        print("ok ->", dst)
    except Exception as e:
        raise SystemExit(f"Could not download {need} ({e}).\n"
                         f"Manually place it at {dst} from https://github.com/UCSC-REAL/cifar-10-100n")
else:
    print("found", dst)

found ..\data\cifar\CIFAR-100_human.pt


In [5]:
# --- build the dataset: images + NOISY human labels (train on these) + CLEAN labels (eval only) ---
# Fast path: the whole CIFAR train set is tiny (50k x 3 x 32 x 32 ~ 586 MB in fp32), so we keep it
# as ONE tensor on the GPU and do augmentation + normalization as batched GPU ops. No DataLoader,
# no CPU workers (this is what was making it slow on Windows).
import torchvision
is10 = (DATASET=="cifar10n")
K = 10 if is10 else 100

base = (torchvision.datasets.CIFAR10 if is10 else torchvision.datasets.CIFAR100)(
    DATA_ROOT, train=True, download=True)
images_np = base.data                      # (50000, 32, 32, 3) uint8
clean  = np.array(base.targets, dtype=np.int64)

human = torch.load(os.path.join(DATA_ROOT, need), map_location="cpu", weights_only=False)
if is10:
    noisy = np.array(human[CIFAR10N_NOISE], dtype=np.int64)
else:
    noisy = np.array(human["noisy_label"], dtype=np.int64)
clean_pt = np.array(human["clean_label"], dtype=np.int64)
assert (clean_pt == clean).mean() > 0.999, "CIFAR-N / torchvision label order mismatch!"
N = len(images_np)
noise_rate = (noisy != clean).mean()
print(f"{DATASET}: N={N:,}  K={K}  real human noise rate={noise_rate:.3%}")

MEAN=torch.tensor([0.4914,0.4822,0.4465]).view(1,3,1,1)
STD =torch.tensor([0.2470,0.2435,0.2616]).view(1,3,1,1)

# all images -> normalized GPU tensor, NCHW, fp32 (augmentation done per-batch below)
X_all = torch.from_numpy(images_np).permute(0,3,1,2).float().div_(255.0)   # (N,3,32,32) cpu
X_all = (X_all - MEAN) / STD
X_all = X_all.to(DEVICE)                                   # resident on GPU
y_noisy = torch.from_numpy(noisy).long().to(DEVICE)
print("images on", X_all.device, "| tensor", tuple(X_all.shape),
      f"| ~{X_all.element_size()*X_all.nelement()/1e6:.0f} MB on GPU")

# pre-pad once for random-crop augmentation (reflect pad 4 on each side -> 40x40)
X_pad = torch.nn.functional.pad(X_all, (4,4,4,4), mode="reflect")          # (N,3,40,40) on GPU

def aug_batch(idx):
    """random 32x32 crop + random horizontal flip, fully on GPU, for a batch of indices.
    Crop indexing verified against a reference loop (channel axis made explicit -> (b,C,32,32))."""
    b = idx.shape[0]; C = X_pad.shape[1]; S = 32
    xs = torch.randint(0,9,(b,),device=DEVICE)   # 0..8 crop offsets (pad=4 each side)
    ys = torch.randint(0,9,(b,),device=DEVICE)
    patches = X_pad[idx]                          # (b,C,40,40)
    ar = torch.arange(S,device=DEVICE)
    rows = (ys.view(b,1)+ar.view(1,S))           # (b,32)
    cols = (xs.view(b,1)+ar.view(1,S))           # (b,32)
    bi = torch.arange(b,device=DEVICE).view(b,1,1,1)
    ci = torch.arange(C,device=DEVICE).view(1,C,1,1)
    out = patches[bi, ci, rows.view(b,1,S,1), cols.view(b,1,1,S)]   # (b,C,32,32)
    flip = torch.rand(b,device=DEVICE) < 0.5     # random horizontal flip
    if flip.any():
        out[flip] = torch.flip(out[flip], dims=[3])
    return out

def eval_batch(idx):
    """no augmentation: just the normalized images."""
    return X_all[idx]

100%|███████████████████████████████████████████████████████████████████████████████| 169M/169M [00:42<00:00, 4.02MB/s]


cifar100n: N=50,000  K=100  real human noise rate=40.200%
images on cuda:0 | tensor (50000, 3, 32, 32) | ~614 MB on GPU


In [6]:
# --- model: ResNet-34 adapted for 32x32 (standard CIFAR stem) ---
import torch.nn as nn, torchvision
def make_resnet(K):
    m = torchvision.models.resnet18(weights=None, num_classes=K)  # resnet18: faster, ample for CIFAR
    # CIFAR stem: 3x3 conv, no maxpool (the ImageNet 7x7/maxpool destroys 32x32 inputs)
    m.conv1 = nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False)
    m.maxpool = nn.Identity()
    return m

In [7]:
# --- k-fold cross-fitting: each item predicted OUT-OF-SAMPLE (no memorization leakage) ---
# Index-based training straight on the GPU-resident tensors. No DataLoader / no CPU workers.
from sklearn.model_selection import StratifiedKFold
pi = np.zeros((N, K), dtype=np.float16)
oos_filled = np.zeros(N, dtype=bool)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

amp = (USE_AMP and DEVICE=="cuda")
try:
    _scaler = torch.amp.GradScaler("cuda", enabled=amp)      # new API
except (AttributeError, TypeError):
    _scaler = torch.cuda.amp.GradScaler(enabled=amp)         # fallback for older torch
ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

def train_fold(model, tr_idx, opt, sched):
    tr = torch.as_tensor(tr_idx, device=DEVICE)
    model.train()
    last=0.0
    for ep in range(EPOCHS):
        perm = tr[torch.randperm(tr.shape[0], device=DEVICE)]
        tot=0.0; m=0
        for s in range(0, perm.shape[0]-BATCH+1, BATCH):   # drop_last
            bidx = perm[s:s+BATCH]
            xb = aug_batch(bidx); yb = y_noisy[bidx]
            with torch.autocast(device_type="cuda", enabled=amp):
                out = model(xb); loss = ce(out, yb)
            opt.zero_grad()
            if amp:
                _scaler.scale(loss).backward(); _scaler.step(opt); _scaler.update()
            else:
                loss.backward(); opt.step()
            sched.step()
            tot+=loss.item()*xb.shape[0]; m+=xb.shape[0]
        last=tot/max(m,1)
    return last

@torch.no_grad()
def predict_oos(model, te_idx):
    model.eval()
    te = torch.as_tensor(te_idx, device=DEVICE)
    for s in range(0, te.shape[0], 512):
        bidx = te[s:s+512]
        with torch.autocast(device_type="cuda", enabled=amp):
            logits = model(eval_batch(bidx))
        p = torch.softmax(logits, dim=1).float().cpu().numpy().astype(np.float16)
        ii = bidx.cpu().numpy()
        pi[ii] = p; oos_filled[ii] = True

import time
for fold,(tr,te) in enumerate(skf.split(np.arange(N), noisy)):
    t0=time.time()
    model=make_resnet(K).to(DEVICE)
    opt=torch.optim.SGD(model.parameters(),lr=LR,momentum=MOMENTUM,
                        weight_decay=WEIGHT_DECAY,nesterov=True)
    steps=(len(tr)//BATCH)*EPOCHS
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(steps,1))
    tl=train_fold(model,tr,opt,sched)
    predict_oos(model,te)
    pred=pi[te].argmax(1)
    print(f"fold {fold}: {time.time()-t0:.0f}s  train_loss={tl:.3f}  "
          f"holdout acc vs noisy={(pred==noisy[te]).mean():.3f}  vs clean={(pred==clean[te]).mean():.3f}")
assert oos_filled.all(), "some items never received an OOS prediction"

fold 0: 224s  train_loss=0.545  holdout acc vs noisy=0.414  vs clean=0.509
fold 1: 223s  train_loss=0.555  holdout acc vs noisy=0.419  vs clean=0.512
fold 2: 223s  train_loss=0.567  holdout acc vs noisy=0.422  vs clean=0.511


In [8]:
# --- save artifacts in the SAME schema as nb01 (so nb12 reads them like AlleNoise) ---
# pi_memmap.npy (float16, N x K), item_meta.parquet, label_classes.npy
pi_path=os.path.join(OUT_DIR,"pi_memmap.npy")
mm=np.lib.format.open_memmap(pi_path,mode="w+",dtype=np.float16,shape=(N,K))
mm[:]=pi[:]; mm.flush(); del mm

EPS=1e-12
pf=pi.astype(np.float32); pf=np.clip(pf,EPS,1.0)
ent=(-(pf*np.log(pf)).sum(1)).astype(np.float32)
p_noisy=pi[np.arange(N),noisy].astype(np.float32)

meta=pd.DataFrame({
    "noisy_id": noisy.astype(int),
    "clean_id": clean.astype(int),            # CIFAR clean labels always known -> no -1
    "is_misregistered": (noisy!=clean).astype(int),
    "p_noisy": p_noisy,
    "entropy": ent,
})
meta.to_parquet(os.path.join(OUT_DIR,"item_meta.parquet"))
np.save(os.path.join(OUT_DIR,"label_classes.npy"), np.arange(K,dtype=np.int64))
with open(os.path.join(OUT_DIR,"meta.txt"),"w") as f:
    f.write(f"dataset={DATASET}\nK={K}\nN={N}\nnoise_rate={noise_rate:.6f}\n"
            f"noise_variant={CIFAR10N_NOISE if is10 else 'noisy_label'}\nfolds={N_FOLDS}\nepochs={EPOCHS}\n")
print(f"saved artifacts to {OUT_DIR}/  (pi_memmap.npy, item_meta.parquet, label_classes.npy, meta.txt)")
print("Now set DATASET to the other CIFAR-N and run again, then run nb12.")

saved artifacts to ..\data\cifar\cifar100n/  (pi_memmap.npy, item_meta.parquet, label_classes.npy, meta.txt)
Now set DATASET to the other CIFAR-N and run again, then run nb12.
